# Notebook Docente 02 — Con nubes vs. sin nubes, y enmascaramiento SCL
## Teledetección — Maestría en Ingeniería, Universidad del Magdalena
**Sesión 2 · Sábado 18 de julio de 2026 · Bloques 1 y 2**

---

**Este notebook es para que TÚ (el docente) lo proyectes en pantalla.** Versión
en Python/Colab del script `docente/gee/02_nubes_y_scl.js`.

### Qué vas a mostrar
La misma zona en temporada seca (limpia) vs. temporada húmeda (con nubes, sin
filtrar a propósito), la banda SCL identificando las nubes, y el efecto real
de calcular NDVI con y sin enmascarar.

In [1]:
!pip install geemap --quiet
print("✓ Instalación completada")

✓ Instalación completada


In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='teledeteccion-miguepoloc')

print("✓ Google Earth Engine inicializado correctamente")

✓ Google Earth Engine inicializado correctamente


## Paso 1 — Imagen seca (referencia) e imagen húmeda (sin filtrar nubes)

In [ ]:
zona_amplia = ee.Geometry.Rectangle([-74.5, 10.2, -73.2, 11.2])

imagen_seca = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_amplia)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(zona_amplia)
)

coleccion_humeda = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_amplia)
    .filterDate('2024-10-01', '2024-11-30')
)

# .mosaic() en vez de .first(): un solo tile Sentinel-2 (~110x110 km) no
# alcanza a cubrir todo zona_amplia (~130x110 km) -- con .first() solo se
# veía el pedacito de un tile (el mismo bug que en 04_optico_vs_sar.ipynb).
# .mosaic() combina TODAS las escenas de oct-nov 2024 que caen en la zona en
# un mosaico que sí cubre el rectángulo completo, sin enmascarar nubes.
imagen_humeda = coleccion_humeda.mosaic().clip(zona_amplia)

n_imagenes_humeda = coleccion_humeda.size().getInfo()
pct_nubes_promedio = coleccion_humeda.aggregate_mean('CLOUDY_PIXEL_PERCENTAGE').getInfo()

print(f'Imágenes Sentinel-2 combinadas en el mosaico oct-nov 2024: {n_imagenes_humeda}')
print(f'% de nubes promedio entre esas imágenes: {pct_nubes_promedio:.1f}%')

## Paso 2 — Enmascarar nubes con la banda SCL

In [4]:
def enmascarar_nubes(imagen):
    scl = imagen.select('SCL')
    # 3=sombra de nube, 8=nube media, 9=nube alta, 10=cirros
    mascara = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return imagen.updateMask(mascara)

imagen_humeda_enmascarada = enmascarar_nubes(imagen_humeda)

ndvi_sin_mascara = imagen_humeda.normalizedDifference(['B8', 'B4']).rename('NDVI_sin_mascara')
ndvi_enmascarado = imagen_humeda_enmascarada.normalizedDifference(['B8', 'B4']).rename('NDVI_enmascarado')

print("✓ Máscara aplicada")

✓ Máscara aplicada


## Paso 3 — Mapa comparativo (activa/desactiva capas)

In [5]:
paleta_scl = ['#000000', '#ff0000', '#2f2f2f', '#643200', '#00a000', '#ffff00',
              '#0000ff', '#bfbfbf', '#808080', '#ffffff', '#66ccff', '#ff66ff']
paleta_ndvi = ['#d73027', '#f46d43', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
vis_color_natural = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}

m = geemap.Map()
m.centerObject(zona_amplia, 10)
m.addLayer(imagen_seca, vis_color_natural, '1 - Color Natural SECA (referencia limpia)')
m.addLayer(imagen_humeda, vis_color_natural, '2 - Color Natural HÚMEDA (con nubes)', False)
m.addLayer(imagen_humeda.select('SCL'), {'min': 0, 'max': 11, 'palette': paleta_scl},
           '3 - SCL (clasificación de escena)', False)
m.addLayer(ndvi_sin_mascara, {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi},
           '4 - NDVI SIN enmascarar (incluye nubes)', False)
m.addLayer(ndvi_enmascarado, {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi},
           '5 - NDVI enmascarado con SCL', False)
m.addLayerControl()
m

Map(center=[10.700397756578358, -73.85], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

## Paso 4 — Cuantificar el problema: % de área válida

In [6]:
area_total = zona_amplia.area().getInfo()

area_valida = (
    ee.Image.pixelArea()
    .updateMask(imagen_humeda_enmascarada.select('B4').mask())
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=zona_amplia, scale=100, maxPixels=1e9)
    .get('area')
    .getInfo()
)

print(f"Área total de la zona: {area_total/1e6:.1f} km²")
print(f"Área válida tras enmascarar SCL: {area_valida/1e6:.1f} km²")
print(f"Porcentaje válido: {100*area_valida/area_total:.1f}%")

Área total de la zona: 15794.6 km²
Área válida tras enmascarar SCL: 13.7 km²
Porcentaje válido: 0.1%


## Guion para la clase

Un pixel de nube en la capa 4 (NDVI sin enmascarar) puede mostrar valores de
0.2-0.4 — el mismo rango que "vegetación escasa". Sin la máscara SCL no podrías
distinguir un pixel de nube real de un pixel de pastizal seco.

**Dato regional:** el Caribe colombiano tiene 60-80% de días nublados en
temporada de lluvias (sept-nov) — de 18 pasajes posibles de Sentinel-2 en 90
días, 12-13 pueden ser inutilizables sin este paso de enmascaramiento.